<a href="https://colab.research.google.com/github/MichalSlowakiewicz/Deep-Neural-Network/blob/master/Exam/task1_2324.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Task 1 (6p)
Your task is to modify the custom implementation of MultiHeadAttention. This custom implementation, currently, enables each token to attent to every other token.


Your job is to change this behavior in a specific way.
Let $S$ be our input sequence of length $2 \cdot k$:
- tokens on positions $i \lt k$ should attend to prefix of $S$ of length $k$ ($S[:k]$) - every token up to position k
- tokens on positions $i \ge k$ should attend to prefix of $S$  of length $i + 1$ ($S[:i + 1]$) - every previous token and itself

(Note: You can assume the sequence length is always an even number).

In [23]:
import torch
import math
import torch.nn.functional as F
class MultiHeadAttention(torch.nn.Module):
    def __init__(self, d_model, num_heads, d_head):
      super().__init__()
      self.d_model = d_model
      self.num_heads = num_heads
      self.d_head = d_head

      self.W_Q = torch.nn.Linear(d_model, num_heads*d_head, bias=True)
      self.W_K = torch.nn.Linear(d_model, num_heads*d_head, bias=True)
      self.W_V = torch.nn.Linear(d_model, num_heads*d_head, bias=True)
      self.W_O = torch.nn.Linear(num_heads*d_head, d_model, bias=True)

    def forward(self, x):

      seq_len, batch_size, _ = x.shape

      Q = self.W_Q(x).reshape(seq_len, batch_size, self.num_heads, self.d_head)
      K = self.W_K(x).reshape(seq_len, batch_size, self.num_heads, self.d_head)
      V = self.W_V(x).reshape(seq_len, batch_size, self.num_heads, self.d_head)

      scaled_QK = torch.einsum("ibhd,jbhd->bhij", Q, K) / math.sqrt(self.d_head)
      # shape of scaled_QK is (batch_size, num_heads, seq_len, seq_len)
      #TODO
      attention_mask  = torch.tril(torch.ones(seq_len, seq_len), diagonal=0).bool()
      v1 = torch.arange(seq_len).unsqueeze(1) # (seq_len, 1)
      v2 = torch.arange(seq_len).unsqueeze(0) # (1, seq_len)
      attention_mask[(v1<seq_len//2) & (v2<seq_len//2)] = True

      inversed_mask = ~attention_mask
      scores = scaled_QK
      scores = scores.masked_fill_(inversed_mask, float('-inf'))
      scaled_QK = scores

      #ENDTODO
      weights = F.softmax(scaled_QK, -1)
      attention = torch.einsum("bhij,jbhd->ibhd", weights, V)

      result = self.W_O(attention.reshape(seq_len, batch_size,self.num_heads * self.d_head))

      return result, weights

In [24]:
# Test your solution
d_model = 10
num_heads= 3
d_head = 2
k = 10
batch_size = 10

mha = MultiHeadAttention(d_model, num_heads, d_head)
batched_x= torch.randn((2*k, batch_size, d_model))
with torch.no_grad():
  result, weights = mha(batched_x)
print("Result:", result)
print("Weights:", weights)

Result: tensor([[[ 0.4469, -0.0386,  0.1229,  ..., -0.5510, -0.0655, -0.2366],
         [ 0.3846, -0.0037,  0.2791,  ..., -0.3810, -0.0182, -0.2536],
         [ 0.3893, -0.0839,  0.1570,  ..., -0.4388, -0.1396, -0.1481],
         ...,
         [ 0.2432, -0.1468,  0.3428,  ..., -0.2425, -0.1819, -0.2205],
         [ 0.2187, -0.0305,  0.2606,  ..., -0.3478, -0.0377, -0.4401],
         [ 0.3763, -0.2197,  0.2254,  ..., -0.4455, -0.2295, -0.0525]],

        [[ 0.4987,  0.0254,  0.1019,  ..., -0.5678, -0.0126, -0.2820],
         [ 0.3183, -0.0847,  0.3102,  ..., -0.3844, -0.0735, -0.1794],
         [ 0.3766, -0.0760,  0.1696,  ..., -0.4739, -0.0978, -0.2353],
         ...,
         [ 0.2281, -0.1438,  0.3747,  ..., -0.2162, -0.1708, -0.2707],
         [ 0.2322, -0.0417,  0.2513,  ..., -0.3570, -0.0525, -0.3874],
         [ 0.3354, -0.2151,  0.2540,  ..., -0.4485, -0.1979, -0.1456]],

        [[ 0.4476, -0.0471,  0.1235,  ..., -0.5489, -0.0753, -0.2122],
         [ 0.3115, -0.0466,  0.2916, 